In [1]:
# 1. IMPORT LIBRARIES & SYSTEM DEPENDENCIES

In [3]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
# 2. DATA INGESTION & DATA CLEANING

In [5]:
CSV_FILE = 'Dataset-SA.csv'

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"Missing file! Please place '{CSV_FILE}' in your running Jupyter directory.")

# Read the dataset using standard latin-1 encoding to prevent decoding crashes
df = pd.read_csv(CSV_FILE, encoding='latin-1')

print("--- Metadata Check ---")
print(f"Total rows detected: {df.shape[0]}")
print(df.isnull().sum())
print("-" * 40)

# HANDLING MISSING TEXT (Crucial Step derived from Metadata Analysis):
# If 'Review' is missing but 'Summary' is present, copy the Summary over.
df['Review'] = df['Review'].fillna(df['Summary'])

--- Metadata Check ---
Total rows detected: 205052
product_name         0
product_price        0
Rate                 0
Review           24664
Summary             11
Sentiment            0
dtype: int64
----------------------------------------


In [6]:
df.head()

,product_name,product_price,Rate,Review,Summary,Sentiment
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive
1,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,awesome,best budget 2 fit cooler nice cooling,positive
2,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,the quality is good but the power of air is de...,positive
3,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,1,useless product,very bad product its a only a fan,negative
4,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,ok ok product,neutral


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205052 entries, 0 to 205051
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   product_name   205052 non-null  object
 1   product_price  205052 non-null  object
 2   Rate           205052 non-null  object
 3   Review         205050 non-null  object
 4   Summary        205041 non-null  object
 5   Sentiment      205052 non-null  object
dtypes: object(6)
memory usage: 9.4+ MB


In [8]:
# Drop remaining rows where both Review and Sentiment values might be missing
df.dropna(subset=['Review', 'Sentiment'], inplace=True)

# Standardize text inputs to string format
df['Review'] = df['Review'].astype(str)

# Clean up Target Column string casings
df['Sentiment'] = df['Sentiment'].str.strip().str.lower()

# Filter out 'neutral' elements to optimize binary classification modeling
# (Optional, but focuses metrics directly onto Positive vs Negative performance)
df = df[df['Sentiment'].isin(['positive', 'negative'])]

print(f"Post-cleaning sample counts:\n{df['Sentiment'].value_counts()}")
print("-" * 40)

Post-cleaning sample counts:
Sentiment
positive    166580
negative     28232
Name: count, dtype: int64
----------------------------------------


In [9]:
# 3. TEXT PREPROCESSING PIPELINE

In [10]:
def clean_review_text(text):
    """
    Cleans e-commerce feedback strings by removing numbers and wild symbols
    while keeping the characters lowercase.
    """
    text = text.lower()
    # Retain only alphabetic text strings and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

print("Applying regex cleaning to review features...")
df['clean_review'] = df['Review'].apply(clean_review_text)

# Map string labels directly to machine coordinates: negative -> 0, positive -> 1
df['target_label'] = df['Sentiment'].map({'negative': 0, 'positive': 1})

Applying regex cleaning to review features...


In [11]:
# 4. STRATIFIED TRAIN-TEST SPLIT

In [12]:
# Use stratify=y to explicitly protect the model against the positive skew bias
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], 
    df['target_label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['target_label']
)

print(f"Training dataset array size: {len(X_train)}")
print(f"Testing dataset array size: {len(X_test)}")
print("-" * 40)

Training dataset array size: 155849
Testing dataset array size: 38963
----------------------------------------


In [13]:
# 5. XG Boost Implementation

In [19]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ----

In [20]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [21]:
num_positive = np.sum(y_train == 1)
num_negative = np.sum(y_train == 0)
imbalance_ratio = num_positive / num_negative

print(f"Calculated scale_pos_weight factor: {imbalance_ratio:.2f}")
print("-" * 50)

Calculated scale_pos_weight factor: 5.90
--------------------------------------------------


In [22]:
xgb_sentiment_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))),
    ('classifier', XGBClassifier(
        scale_pos_weight=imbalance_ratio,
        n_estimators=150,          # Number of sequential trees to build
        max_depth=6,               # Tree depth limitation to prevent overfitting
        learning_rate=0.1,         # Step size shrinkage used to prevent overfitting
        random_state=42,
        n_jobs=-1                  # Use all available CPU cores for maximum speed
    ))
])

In [23]:
print("Training the XGBoost Ensemble Pipeline (this might take a minute)...")
xgb_sentiment_pipeline.fit(X_train, y_train)
print("🎯 Ensemble model trained successfully!")
print("-" * 50)

# Run evaluation pass
xgb_predictions = xgb_sentiment_pipeline.predict(X_test)

print(f"XGBoost Overall Accuracy: {accuracy_score(y_test, xgb_predictions) * 100:.2f}%\n")
print("XGBoost Detailed Classification Report:")
print(classification_report(y_test, xgb_predictions, target_names=['Negative', 'Positive']))

Training the XGBoost Ensemble Pipeline (this might take a minute)...
🎯 Ensemble model trained successfully!
--------------------------------------------------
XGBoost Overall Accuracy: 94.54%

XGBoost Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.93      0.67      0.78      5646
    Positive       0.95      0.99      0.97     33317

    accuracy                           0.95     38963
   macro avg       0.94      0.83      0.88     38963
weighted avg       0.94      0.95      0.94     38963



In [24]:
import joblib

model_filename = 'flipkart_xgboost_sentiment_model.pkl'
joblib.dump(xgb_sentiment_pipeline, model_filename)

print(f"💾 XGBoost Pipeline successfully saved as: {model_filename}")

💾 XGBoost Pipeline successfully saved as: flipkart_xgboost_sentiment_model.pkl
